In [3]:
import pandas as pd 
df = pd.read_csv("Animal_Dataset.csv") #loads in csv file into dataframe 

### New animal request

-get user to fill out a csv file with appropriate stats in appropriate format
-manual checks for other stats and images on site


#### Image, map & food chain check

In [4]:
import requests
import os
import shutil


HEADERS = {
    "User-Agent": "AnimalImageFetcher/1.0 (your_email@example.com)"
}

WIKI_API = "https://en.wikipedia.org/w/api.php"

LOCAL_IMAGE_MAP = {     #local manually added images
    "Blobfish": "static/images/Blobfish.jpeg",
    "Fossa": "static/images/Fossa.jpeg",
    "Kiwi": "static/images/kiwi.jpeg",
    "Red-Eyed Tree Frog": "static/images/red-eye tree frog.jpeg",
    "Amazon Rainforest Frog": "static/images/Amz rainforest frog.jpeg",
    "Titanaboa": "static/images/Titanoboa.jpeg",
    "Woolly Mammoth": "static/images/woolly mammoth.jpeg",
}

def resolve_title(animal_name):     #uses full search engine (searches multiple articles)
    params = {
        "action": "query",
        "list": "search",
        "srsearch": animal_name,
        "format": "json"
    }

    r = requests.get(WIKI_API, params=params, headers=HEADERS)
    r.raise_for_status()
    results = r.json().get("query", {}).get("search", [])

    #return top ranked page in wiki
    return results[0]["title"] if results else None


def get_animal_image(animal_name):
    
    #check if it's manually added
    if animal_name in LOCAL_IMAGE_MAP:
        path = LOCAL_IMAGE_MAP[animal_name]
        if os.path.exists(path):
            return path   # local file path
        else:
            print(f"Missing local image file: {path}")
    
    #look for image directly
    params = {
        "action": "query",
        "titles": animal_name,
        "prop": "pageimages",
        "format": "json",
        "pithumbsize": 1000,
        "redirects": 1      #allows page redirect
    }

    r = requests.get(WIKI_API, params=params, headers=HEADERS)
    r.raise_for_status()
    page = next(iter(r.json()["query"]["pages"].values()))

    if "thumbnail" in page:
        return page["thumbnail"]["source"]

    # Search fallback
    resolved = resolve_title(animal_name)       #try another page
    if not resolved:
        return None

    params["titles"] = resolved     #get image from other page
    r = requests.get(WIKI_API, params=params, headers=HEADERS)
    r.raise_for_status()
    page = next(iter(r.json()["query"]["pages"].values()))

    return page.get("thumbnail", {}).get("source")

def download_image(image_source, filename):
    if not image_source:
        return

    # Wikipedia / remote image
    if image_source.startswith("http"):
        response = requests.get(image_source, headers=HEADERS)
        if response.status_code == 200:
            with open(filename, "wb") as f:
                f.write(response.content)

    # Local image path
    else:
        if os.path.exists(image_source):
            shutil.copy(image_source, filename)
        else:
            print(f"Local image not found: {image_source}")

# ---- Test ----
animal = "Jaguar"
image_url = get_animal_image(animal)

if image_url:
    print("Image found:", image_url)
    download_image(image_url, f"{animal}.jpg")
else:
    print("No image found")

Image found: https://upload.wikimedia.org/wikipedia/commons/0/0a/Standing_jaguar.jpg


In [6]:
import geopandas as gpd
import matplotlib.pyplot as plt
from shapely.geometry import box
import re

WORLD_URL = ("https://naturalearth.s3.amazonaws.com/"
            "110m_cultural/ne_110m_admin_0_countries.zip")

OCEAN_URL = ("https://naturalearth.s3.amazonaws.com/"
            "50m_physical/ne_50m_geography_marine_polys.zip")

# --- Custom Region Groups ---
REGION_ALIASES = {
    "americas": ["North America", "South America"],
    "amazon rainforest": ["Brazil", "Peru", "Bolivia", "Ecuador", 
                        "Colombia", "Venezuela", "French Guiana", "Guyana", "Suriname"],
    "central america": ["Mexico", "Guatemala", "Belize", "Honduras",
                        "El Salvador", "Nicaragua", "Costa Rica", "Panama"],
    "middle east": ["Saudi Arabia", "Iran", "Iraq", "Israel", "Jordan",
                    "United Arab Emirates", "Qatar", "Kuwait",
                    "Oman", "Yemen", "Syria", "Lebanon"],
    "indian subcontinent": ["India", "Pakistan", "Bangladesh", "Nepal",
                            "Bhutan", "Sri Lanka"],
    "north africa": ["Africa"],
    "indonesia": ["Indonesia"],
    "borneo": ["Indonesia", "Malaysia", "Brunei"],
    "sumatra": ["Indonesia"],
    "southeast asia": ["Asia"],
    "south asia": ["Asia"],
    "tasmania": ["Australia"],
    "galápagos islands": ["Ecuador"],
    "new guinea": ["Papua New Guinea", "Indonesia"],
    "sub-saharan africa": ["Africa"],  # simplified
    "indo-pacific region": ["Indian Ocean", "Pacific Ocean"],
    "all oceans": ["Pacific Ocean", "Atlantic Ocean",
                "Indian Ocean", "Arctic Ocean", "Southern Ocean"],
    "oceans worldwide": ["Pacific Ocean", "Atlantic Ocean",
                        "Indian Ocean", "Arctic Ocean", "Southern Ocean"],
    "worldwide": ["Africa", "Asia", "Europe",
                "North America", "South America", "Oceania", "Antarctica"],
    "northern hemisphere": ["North America", "Europe", "Asia"],
}

# Words to remove (ignore cardinal directions)
REMOVE_WORDS = ["western", "eastern", "northern", "southern",
    "central", "tropical"]


def clean_region_name(name):
    name = name.lower()

    # remove direction words
    for word in REMOVE_WORDS:
        name = re.sub(rf"\b{word}\b", "", name)

    name = name.replace("(", "").replace(")", "")
    name = name.strip()

    return name


def highlight_region(input_string, show_map=True):

    world = gpd.read_file(WORLD_URL)
    world.columns = world.columns.str.lower()
    oceans = gpd.read_file(OCEAN_URL)
    oceans.columns = oceans.columns.str.lower()

    regions = [r.strip() for r in input_string.split(",")]

    matches = []

    for region_raw in regions:

        region = clean_region_name(region_raw)

        # --- Alias group ---
        if region in REGION_ALIASES:
            for subregion in REGION_ALIASES[region]:

                # Ocean inside alias
                ocean_match = oceans[oceans["name_en"].str.lower() == subregion.lower()]
                
                if not ocean_match.empty:
                    matches.append(ocean_match)
                    continue

                # Continent inside alias
                continent_match = world[world["continent"].str.lower() == subregion.lower()]
                
                if not continent_match.empty:
                    matches.append(continent_match)
                    continue

                # Country inside alias
                country_match = world[world["admin"].str.lower() == subregion.lower()]
                
                if not country_match.empty:
                    matches.append(country_match)
            continue

        # --- Continent ---
        continent_match = world[world["continent"].str.lower() == region]
        
        if not continent_match.empty:
            matches.append(continent_match)
            continue

        # --- Country ---
        country_match = world[world["admin"].str.lower() == region]
        
        if not country_match.empty:
            matches.append(country_match)
            continue

        # --- Ocean ---
        ocean_match = oceans[oceans["name_en"].str.lower() == region]
        
        if not ocean_match.empty:
            matches.append(ocean_match)
            continue

        # --- Arctic ---
        if region == "arctic":
            arctic_box = box(-180, 66.5, 180, 90)
            arctic_gdf = gpd.GeoDataFrame(geometry=[arctic_box], crs=world.crs)
            matches.append(arctic_gdf)
            continue

        if not matches:
            return -1

    if show_map:
        fig, ax = plt.subplots(figsize=(12, 6))

        world.plot(ax=ax, color="lightgray", edgecolor="white")
        oceans.plot(ax=ax, color="lightblue", edgecolor="lightblue")

        for match in matches:
            match.plot(ax=ax, color="green", edgecolor="black", alpha=0.7)

        ax.set_title(f"Highlighted: {input_string}")
        ax.axis("off")
        plt.show()

#highlight_region('Russia, China')


In [7]:
def check_images_for_animal(df, animal_name):
    result = df[df["Animal"].str.lower() == animal_name.lower()]
    if result.empty:
        print(f"Animal '{animal_name}' not found in dataset")
        return

    row = result.iloc[0]
    missing = []

    # check foods
    foods = row["Food"]
    if isinstance(foods, str):
        foods = foods.strip("()")
        foods = [f.strip().lower() for f in foods.split(",") if f.strip()]
        for food in foods:
            if get_animal_image(food) is None:
                missing.append(("food", food))

    # check predators
    predators = row["Predators"]
    if isinstance(predators, str) and predators.strip().lower() != "not applicable":
        predators = [p.strip().lower() for p in predators.split(",") if p.strip()]
        for predator in predators:
            if get_animal_image(predator) is None:
                missing.append(("predator", predator))

    if missing:
        for category, name in missing:
            print(f"[{category}] '{name}' has no image")
    else:
        print(f"All images found for {animal_name}")

    return missing

check_images_for_animal(df, "Aardvark")

All images found for Aardvark


[]

# Run this (for actual addition)

In [8]:
###FINAL CHECK 

def validate_new_animal(df, animal_name):
    result = df[df["Animal"].str.lower() == animal_name.lower()]
    if result.empty:
        print(f"Animal '{animal_name}' not found in dataset")
        return

    row = result.iloc[0]
    missing = []

    # check animal itself
    if get_animal_image(animal_name) is None:
        missing.append(("animal", animal_name))

    # check foods
    foods = row["Food"]
    if isinstance(foods, str):
        foods = foods.strip("()")
        foods = [f.strip().lower() for f in foods.split(",") if f.strip()]
        for food in foods:
            if get_animal_image(food) is None:
                missing.append(("food", food))

    # check predators
    predators = row["Predators"]
    if isinstance(predators, str) and predators.strip().lower() != "not applicable":
        predators = [p.strip().lower() for p in predators.split(",") if p.strip()]
        for predator in predators:
            if get_animal_image(predator) is None:
                missing.append(("predator", predator))

    # check countries found
    countries = row["Countries Found"]
    if isinstance(countries, str):
        map_result = highlight_region(countries, show_map=False)
        if map_result == -1:
            missing.append(("countries", countries))

    if missing:
        for category, name in missing:
            print(f"[{category}] '{name}' has no image or map match")
    else:
        print(f"All checks passed for {animal_name}")

    return missing

df1 = pd.read_csv("Added_to_data.csv") #loads in csv file into dataframe 

validate_new_animal(df1, "Giraffe")

All checks passed for Giraffe


[]

#### Automate integration

# Run this (for actual addition)

In [9]:
def add_animal_to_dataset(animal_name):
    new_df = pd.read_csv("Added_to_data.csv")
    
    result = new_df[new_df["Animal"].str.lower() == animal_name.lower()]
    if result.empty:
        print(f"'{animal_name}' not found in Added_to_data.csv")
        return

    original_df = pd.read_csv("Animal_Dataset.csv")

    # check if already exists
    already_exists = original_df[original_df["Animal"].str.lower() == animal_name.lower()]
    if not already_exists.empty:
        print(f"'{animal_name}' already exists in Animal_Dataset.csv")
        return

    # append and sort
    combined = pd.concat([original_df, result], ignore_index=True)
    combined = combined.sort_values("Animal", key=lambda x: x.str.lower())
    combined = combined.reset_index(drop=True)

    combined.to_csv("Animal_Dataset.csv", index=False) 
    print(f"'{animal_name}' successfully added to Animal_Dataset.csv")   

add_animal_to_dataset("Giraffe")

'Giraffe' successfully added to Animal_Dataset.csv


In [10]:
def remove_animal_from_dataset(animal_name):
    original_df = pd.read_csv("Animal_Dataset.csv")

    result = original_df[original_df["Animal"].str.lower() == animal_name.lower()]
    if result.empty:
        print(f"'{animal_name}' not found in Animal_Dataset.csv")
        return

    updated_df = original_df[original_df["Animal"].str.lower() != animal_name.lower()]
    updated_df = updated_df.reset_index(drop=True)

    updated_df.to_csv("Animal_Dataset.csv", index=False)
    print(f"'{animal_name}' successfully removed from Animal_Dataset.csv")


remove_animal_from_dataset("Giraffe")

'Giraffe' successfully removed from Animal_Dataset.csv


### Data change request

-get user to get what they want changed in a textbox (tell them to specify which animal to change, which columns to change, what new values they want and an 
explanation for why they want it changed)